# Modelo relacional da SciELO (2026)

Scripts para converter a base de trabalhos da SciELO em um modelo relacional. Mesmo SQL do insyspo/scielo (`SciELO_relational_model_2024.ipynb`), reexecutado com o dump de 2026 no projeto `multiobs`.

O passo anterior é converter os arquivos XML em objetos JSON e carregá-los numa única tabela do BigQuery, com uma coluna contendo o JSON como texto. Um trabalho por linha (`SciELO_upload_2026.ipynb`).

Diferenças em relação ao notebook de 2024:

- Projeto `insyspo` -> `multiobs`.
- Schema `insyspo.publicdb_scielo_2024_rm` -> `multiobs.publicdb_scielo_2026_09_rm`. Tabela de entrada `insyspo.publicdb_scielo_2024.works_json` -> `multiobs.publicdb_scielo_2026_09.works_json` (`SciELO_upload_2026.ipynb`).
- Uma exportação para uma base de referência foi removida.
- **Correção na `works`**: o JOIN com `journals` era `a.issn LIKE CONCAT('%',b.issn,'%')`. Como `journals.issn` é uma lista (`STRING_AGG`), os trabalhos de revistas com mais de um ISSN eram perdidos (61.208 trabalhos no dump de 2026). Agora é `b.issn LIKE CONCAT('%',a.issn,'%')`.
- Algumas referências no notebook de 2024 apontavam para o dataset antigo de 2023, `insyspo.projectdb_scielo_rm` (a `authors_references` na `works_authors_references` e o `DROP TABLE` final da `keywords`). Agora elas apontam para as tabelas novas.
- Estatísticas estruturais: a expressão regular foi adaptada para JSON com aspas duplas.

## Preparation

In [1]:
import pandas as pd
import google.auth
from google.cloud import bigquery

In [2]:
creds, _ = google.auth.default()
creds = creds.with_quota_project(None)
bq_client = bigquery.Client(project='multiobs', credentials=creds)

In [3]:
%load_ext bigquery_magics
import bigquery_magics
bigquery_magics.context.credentials = creds
bigquery_magics.context.project = 'multiobs'

## Structural statistics

In [4]:
%%bigquery --project=multiobs

SELECT  SUM(ARRAY_LENGTH(REGEXP_EXTRACT_ALL(works, r'"([^"]+)":'))) AS all_fields,
  AVG(ARRAY_LENGTH(REGEXP_EXTRACT_ALL(works, r'"([^"]+)":'))) AS mean_fields
FROM `multiobs.publicdb_scielo_2026_09.works_json`

Query is running:   0%|          |

Downloading:   0%|          |

,all_fields,mean_fields
0,675767365,528.253603


In [5]:
%%bigquery --project=multiobs

CREATE SCHEMA IF NOT EXISTS multiobs.publicdb_scielo_2026_09_rm

Query is running:   0%|          |

""


## Journals

In [6]:
%%bigquery --project=multiobs

DROP TABLE IF EXISTS `multiobs.publicdb_scielo_2026_09_rm.journals`;
CREATE TABLE `multiobs.publicdb_scielo_2026_09_rm.journals` AS (
  WITH journals_aux AS (
  SELECT
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.front.journal-meta.journal-id['#text']"),'"') AS journal_id,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.front.journal-meta.journal-id['@journal-id-type']"),'"') AS journal_type,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.front.journal-meta.journal-title-group.journal-title"),'"') AS journal_title,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.front.journal-meta.journal-title-group.abbrev-journal-title"),'"') AS abbrev_journal,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.front.journal-meta.issn"),'"') AS issn,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.front.journal-meta.collection"),'"') AS collection,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.front.journal-meta.publisher.publisher-name"),'"') AS publisher,
  FROM `multiobs.publicdb_scielo_2026_09.works_json`
  )
  SELECT
    ROW_NUMBER() OVER() AS jid,
    ANY_VALUE(journal_id) AS label,
    ANY_VALUE(journal_type) AS type,
    journal_title AS title,
    ANY_VALUE(abbrev_journal) AS abbrev,
    STRING_AGG(DISTINCT(issn), ', ') AS issn,
    ANY_VALUE(collection) AS collection,
    ANY_VALUE(publisher) AS publisher,
  FROM journals_aux
  GROUP BY title
);

Query is running:   0%|          |

""


In [7]:
%%bigquery --project=multiobs

SELECT column_name
FROM multiobs.publicdb_scielo_2026_09_rm.INFORMATION_SCHEMA.COLUMNS
WHERE table_name LIKE 'journals'

Query is running:   0%|          |

Downloading:   0%|          |

,column_name
0,jid
1,label
2,type
3,title
4,abbrev
5,issn
6,collection
7,publisher


## Works

Table with information about works. The original dump contains repetitions because there is overlapping of works in the field `collection`.

In [8]:
%%bigquery --project=multiobs

DROP TABLE IF EXISTS `multiobs.publicdb_scielo_2026_09_rm.r_works`;
CREATE TABLE `multiobs.publicdb_scielo_2026_09_rm.r_works` AS (
WITH repeated_works AS (
SELECT  TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.unique-article-id['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id[0]['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id['#text']")
        ),'"') AS id,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.front.article-meta.article-id[1]['#text']"),'"') AS doi,
        TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.title-group.article-title['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.title-group.article-title[0]['#text']")
        ),'"') AS title,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article['@lang_id']"),'"') AS lang,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article['@article-type']"),'"') AS type,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.front.journal-meta.issn"),'"') AS issn,
        COALESCE(
          CAST(TRIM(JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta['date']['year']"),'"') AS INT),
          CAST(TRIM(JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta['pub-date']['year']"),'"') AS INT)) AS pub_year,
FROM `multiobs.publicdb_scielo_2026_09.works_json`
GROUP BY id, doi, title, lang, type, issn, pub_year
)
SELECT *
FROM repeated_works
WHERE id IS NOT NULL
);

DROP TABLE IF EXISTS `multiobs.publicdb_scielo_2026_09_rm.works`;
CREATE TABLE `multiobs.publicdb_scielo_2026_09_rm.works` AS (
WITH aux AS (
  SELECT id,
    doi,
    issn,
    title,
    lang,
    type,
    pub_year,
    ROW_NUMBER() OVER (PARTITION BY id ORDER BY id DESC) r,
  FROM `multiobs.publicdb_scielo_2026_09_rm.r_works`
  ORDER BY id
  )
  SELECT a.id,
      a.doi,
      a.issn,
      a.title,
      a.lang,
      a.type,
      a.pub_year,
      b.jid
  FROM aux AS a
  JOIN multiobs.publicdb_scielo_2026_09_rm.journals AS b
  ON b.issn LIKE CONCAT('%',a.issn,'%')
  WHERE r = 1 AND id IS NOT NULL
  ORDER BY r DESC, id
);

DROP TABLE IF EXISTS `multiobs.publicdb_scielo_2026_09_rm.r_works`;

Query is running:   0%|          |

""


In [9]:
%%bigquery --project=multiobs

SELECT column_name
FROM multiobs.publicdb_scielo_2026_09_rm.INFORMATION_SCHEMA.COLUMNS
WHERE table_name LIKE 'works'

Query is running:   0%|          |

Downloading:   0%|          |

,column_name
0,id
1,doi
2,issn
3,title
4,lang
5,type
6,pub_year
7,jid


In [10]:
%%bigquery --project=multiobs

SELECT COUNT(*) AS n, COUNT(DISTINCT(id)) AS nid
FROM `multiobs.publicdb_scielo_2026_09_rm.works`;

Query is running:   0%|          |

Downloading:   0%|          |

,n,nid
0,1252280,1236917


## Bibliographic

In [11]:
%%bigquery --project=multiobs

DROP TABLE IF EXISTS `multiobs.publicdb_scielo_2026_09_rm.works_biblio`;
CREATE TABLE `multiobs.publicdb_scielo_2026_09_rm.works_biblio` AS (
WITH works AS (
SELECT  TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.unique-article-id['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id[0]['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id['#text']")
        ),'"') AS id,
        TRIM(JSON_EXTRACT(
          works,
          '$.articles.article.front.journal-meta.collection'),'"') AS collection,
        COALESCE(
          CAST(TRIM(JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta['pub-date']['year']"),'"') AS INT),
          CAST(TRIM(JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta['date']['year']"),'"') AS INT))
             AS pub_year,
        COALESCE(
          CAST(TRIM(JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta['pub-date']['month']"),'"') AS INT),
          CAST(TRIM(JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta['date']['month']"),'"') AS INT))
             AS pub_month,
        TRIM(JSON_EXTRACT(
          works,
          '$.articles.article.front.article-meta.volume'),'"') AS volume,
        TRIM(JSON_EXTRACT(
          works,
          '$.articles.article.front.article-meta.issue'),'"') AS issue,
        TRIM(JSON_EXTRACT(
          works,
          '$.articles.article.front.article-meta.lpage'),'"') AS lpage,
        TRIM(JSON_EXTRACT(
          works,
          '$.articles.article.front.article-meta.fpage'),'"') AS fpage,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.front.article-meta.permissions.license['@license-type']"),'"') AS license,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.front.article-meta.self-uri"),'"') AS uris,
FROM `multiobs.publicdb_scielo_2026_09.works_json`
)
SELECT DISTINCT *
FROM works
WHERE id IS NOT NULL
);

Query is running:   0%|          |

""


In [12]:
%%bigquery --project=multiobs

SELECT column_name
FROM multiobs.publicdb_scielo_2026_09_rm.INFORMATION_SCHEMA.COLUMNS
WHERE table_name LIKE 'works_biblio'

Query is running:   0%|          |

Downloading:   0%|          |

,column_name
0,id
1,collection
2,pub_year
3,pub_month
4,volume
5,issue
6,lpage
7,fpage
8,license
9,uris


## Abstracts

In [13]:
%%bigquery --project=multiobs

DROP TABLE IF EXISTS `multiobs.publicdb_scielo_2026_09_rm.works_abstracts`;
CREATE TABLE `multiobs.publicdb_scielo_2026_09_rm.works_abstracts` AS (
SELECT DISTINCT TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.unique-article-id['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id[0]['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id['#text']")
        ),'"') AS id,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.front.article-meta.abstract['@lang_id']"),'"') AS lang_id,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.front.article-meta.abstract.p"),'"') AS text,
FROM `multiobs.publicdb_scielo_2026_09.works_json`
WHERE JSON_EXTRACT(
          works,
          "$.articles.article.front.article-meta.abstract.p") IS NOT NULL
);

Query is running:   0%|          |

""


In [14]:
%%bigquery --project=multiobs

SELECT column_name
FROM multiobs.publicdb_scielo_2026_09_rm.INFORMATION_SCHEMA.COLUMNS
WHERE table_name LIKE 'works_abstracts'

Query is running:   0%|          |

Downloading:   0%|          |

,column_name
0,id
1,lang_id
2,text


## Subjects

In [15]:
%%bigquery --project=multiobs

DROP TABLE IF EXISTS multiobs.publicdb_scielo_2026_09_rm.works_subjects;
CREATE TABLE multiobs.publicdb_scielo_2026_09_rm.works_subjects AS (
  WITH subjects AS (
    SELECT TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.unique-article-id['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id[0]['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id['#text']")
        ),'"') AS id,
            JSON_EXTRACT_ARRAY(
              works,
              "$.articles.article.front.article-meta.article-categories.subj-group.subject") AS subjs,
     FROM `multiobs.publicdb_scielo_2026_09.works_json`
     WHERE JSON_EXTRACT_ARRAY(
              works,
              "$.articles.article.front.article-meta.article-categories.subj-group.subject") IS NOT NULL
  )
  SELECT id,
    TRIM(s,'"') AS subject,
    off+1 AS subject_order
  FROM subjects, UNNEST(subjs) AS s WITH OFFSET AS off
  WHERE id IS NOT NULL AND subjs IS NOT NULL
  UNION ALL
      SELECT TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.unique-article-id['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id[0]['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id['#text']")
        ),'"') AS id,
        TRIM(JSON_EXTRACT(
              works,
              "$.articles.article.front.article-meta.article-categories.subj-group.subject"),'"') AS subject,
        1 AS subject_order
     FROM `multiobs.publicdb_scielo_2026_09.works_json`
     WHERE JSON_EXTRACT_ARRAY(
              works,
              "$.articles.article.front.article-meta.article-categories.subj-group.subject") IS NULL
)

Query is running:   0%|          |

""


In [16]:
%%bigquery --project=multiobs

SELECT column_name
FROM multiobs.publicdb_scielo_2026_09_rm.INFORMATION_SCHEMA.COLUMNS
WHERE table_name LIKE 'works_subjects'

Query is running:   0%|          |

Downloading:   0%|          |

,column_name
0,id
1,subject
2,subject_order


## Authors

In [17]:
%%bigquery --project=multiobs

DROP TABLE IF EXISTS `multiobs.publicdb_scielo_2026_09_rm.works_authors`;
CREATE TABLE `multiobs.publicdb_scielo_2026_09_rm.works_authors` AS (
  WITH authors_fjs AS (
    SELECT TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.unique-article-id['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id[0]['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id['#text']")
        ),'"') AS id,
          JSON_EXTRACT_ARRAY(
            works,
            "$.articles.article.front.article-meta.contrib-group.contrib") AS authors
    FROM `multiobs.publicdb_scielo_2026_09.works_json`
    WHERE JSON_EXTRACT_ARRAY(
            works,
            "$.articles.article.front.article-meta.contrib-group.contrib") IS NOT NULL
  )
  SELECT DISTINCT TRIM(id,'"') AS id,
    TRIM(JSON_EXTRACT(author,"$.name.surname"),'"') AS surname,
    TRIM(JSON_EXTRACT(author,"$.name.given-names"),'"') AS given_names,
    TRIM(JSON_EXTRACT(author,"$.xref['@rid']"),'"') AS ref_affiliation,
    off+1 AS author_order
  FROM authors_fjs,
  UNNEST(authors) author WITH OFFSET AS off
  UNION ALL
      SELECT TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.unique-article-id['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id[0]['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id['#text']")
        ),'"') AS id,
          TRIM(JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.contrib-group.contrib.name.surname"),'"') AS surname,
          TRIM(JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.contrib-group.contrib.name.given-names"),'"') AS given_names,
          TRIM(JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.contrib-group.contrib.xref['@rid']"),'"') AS ref_affiliation,
            1 AS author_order
    FROM `multiobs.publicdb_scielo_2026_09.works_json`
    WHERE JSON_EXTRACT_ARRAY(
            works,
            "$.articles.article.front.article-meta.contrib-group.contrib") IS NULL
);

Query is running:   0%|          |

""


In [18]:
%%bigquery --project=multiobs

SELECT column_name
FROM multiobs.publicdb_scielo_2026_09_rm.INFORMATION_SCHEMA.COLUMNS
WHERE table_name LIKE 'works_authors'

Query is running:   0%|          |

Downloading:   0%|          |

,column_name
0,id
1,surname
2,given_names
3,ref_affiliation
4,author_order


## Affiliations

In [19]:
%%bigquery --project=multiobs

DROP TABLE IF EXISTS `multiobs.publicdb_scielo_2026_09_rm.works_affiliations`;
CREATE TABLE `multiobs.publicdb_scielo_2026_09_rm.works_affiliations` AS (
  WITH affiliations_fjs AS (
    SELECT TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.unique-article-id['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id[0]['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id['#text']")
        ),'"') AS id,
          JSON_EXTRACT_ARRAY(
            works,
            '$.articles.article.front.article-meta.aff') AS affiliations
    FROM `multiobs.publicdb_scielo_2026_09.works_json`
    WHERE JSON_EXTRACT_ARRAY(
            works,
            '$.articles.article.front.article-meta.aff') IS NOT NULL
  )
  SELECT DISTINCT TRIM(id,'"') AS id,
    TRIM(JSON_EXTRACT(affiliation,"$['@id']"),'"') AS ref_affiliation,
    TRIM(JSON_EXTRACT(affiliation,"$.institution"),'"') AS institution,
    TRIM(JSON_EXTRACT(affiliation,"$.country"),'"') AS country,
    off+1 AS affiliation_order
  FROM affiliations_fjs,
  UNNEST(affiliations) affiliation WITH offset AS off
  WHERE TRIM(JSON_EXTRACT(affiliation,"$.institution"),'"') IS NOT NULL
      AND TRIM(JSON_EXTRACT(affiliation,"$.institution"),'"') != 'None'
  UNION ALL
  SELECT TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.unique-article-id['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id[0]['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id['#text']")
        ),'"') AS id,
        TRIM(JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.aff['@id']"),'"') AS ref_affiliation,
        TRIM( JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.aff.institution"),'"') AS institution,
        TRIM(JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.aff.country"),'"') AS country,
          1 AS affiliation_order
  FROM `multiobs.publicdb_scielo_2026_09.works_json`
  WHERE JSON_EXTRACT_ARRAY(
            works,
            '$.articles.article.front.article-meta.aff') IS NULL AND
        JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.aff.institution") IS NOT NULL
            AND
        TRIM(JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.aff.institution"),'"') != 'None'
);

Query is running:   0%|          |

""


In [20]:
%%bigquery --project=multiobs

SELECT column_name
FROM multiobs.publicdb_scielo_2026_09_rm.INFORMATION_SCHEMA.COLUMNS
WHERE table_name LIKE 'works_affiliations'

Query is running:   0%|          |

Downloading:   0%|          |

,column_name
0,id
1,ref_affiliation
2,institution
3,country
4,affiliation_order


## References

Works referenced. They are usually not part of Scielo.

In [21]:
%%bigquery --project=multiobs

DROP TABLE IF EXISTS `multiobs.publicdb_scielo_2026_09_rm.works_references`;
CREATE TABLE `multiobs.publicdb_scielo_2026_09_rm.works_references` AS (
  WITH references_fjs AS (
    SELECT TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.unique-article-id['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id[0]['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id['#text']")
        ),'"') AS id,
        JSON_EXTRACT_ARRAY(
          works,
          "$.articles.article.back.ref-list.ref") AS references
    FROM `multiobs.publicdb_scielo_2026_09.works_json`
    WHERE JSON_EXTRACT_ARRAY(
          works,
          "$.articles.article.back.ref-list.ref") IS NOT NULL
  )
  SELECT DISTINCT TRIM(id,'"') AS id,
    TRIM(JSON_EXTRACT(reference,"$['@id']"),'"') AS ref_citation,
    TRIM(JSON_EXTRACT(reference,"$.element-citation['@publication-type']"),'"') AS type,
    TRIM(JSON_EXTRACT(reference,"$.element-citation.source"),'"') AS source,
    TRIM(JSON_EXTRACT(reference,"$.element-citation.article-title"),'"') AS title,
    TRIM(JSON_EXTRACT(reference,"$.element-citation.date.year"),'"') AS year,
    #TRIM(JSON_EXTRACT(reference,'$.element-citation.person-group'),'"') AS authors,
    #TRIM(JSON_EXTRACT(reference,"$.element-citation.ext-link"),'"') AS ext_link,
    TRIM(JSON_EXTRACT(reference,"$.element-citation.ext-link['@href']"),'"') AS uri,
    off+1 AS reference_order
  FROM references_fjs,
  UNNEST(references) reference WITH OFFSET AS off
  UNION ALL
  SELECT TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.unique-article-id['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id[0]['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id['#text']")
        ),'"') AS id,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.back.ref-list.ref['@id']"),'"') AS ref_citation,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.back.ref-list.ref.element-citation['@publication-type']"),'"') AS type,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.back.ref-list.ref.element-citation.source"),'"') AS source,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.back.ref-list.ref.element-citation.article-title"),'"') AS title,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.back.ref-list.ref.element-citation.date.year"),'"') AS year,
        TRIM(JSON_EXTRACT(
          works,
          "$.articles.article.back.ref-list.ref.element-citation.ext-link['@href']"),'"') AS uri,
        1 AS reference_order,
    FROM `multiobs.publicdb_scielo_2026_09.works_json`
    WHERE JSON_EXTRACT_ARRAY(
          works,
          "$.articles.article.back.ref-list.ref") IS NULL
);

Query is running:   0%|          |

""


In [22]:
%%bigquery --project=multiobs

SELECT column_name
FROM multiobs.publicdb_scielo_2026_09_rm.INFORMATION_SCHEMA.COLUMNS
WHERE table_name LIKE 'works_references'

Query is running:   0%|          |

Downloading:   0%|          |

,column_name
0,id
1,ref_citation
2,type
3,source
4,title
5,year
6,uri
7,reference_order


## Authors of references

In [23]:
%%bigquery --project=multiobs

DROP TABLE IF EXISTS `multiobs.publicdb_scielo_2026_09_rm.authors_references`;
CREATE TABLE `multiobs.publicdb_scielo_2026_09_rm.authors_references` AS (
    WITH references_fjs AS (
      SELECT TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.unique-article-id['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id[0]['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id['#text']")
        ),'"') AS id,
            JSON_EXTRACT_ARRAY(
              works,
              "$.articles.article.back.ref-list.ref") AS references
      FROM `multiobs.publicdb_scielo_2026_09.works_json`
      WHERE JSON_EXTRACT_ARRAY(
              works,
              "$.articles.article.back.ref-list.ref") IS NOT NULL
    )
    SELECT id,
      TRIM(JSON_EXTRACT(reference,"$['@id']"),'"') AS ref_citation,
      JSON_EXTRACT(reference,"$.element-citation.person-group.name") AS authors,
      off+1 AS reference_order
    FROM references_fjs,
    UNNEST(references) reference WITH OFFSET AS off
    UNION ALL
    SELECT TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.unique-article-id['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id[0]['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id['#text']")
        ),'"') AS id,
        TRIM(JSON_EXTRACT(
              works,
              "$.articles.article.back.ref-list.ref['@id']"),'"') AS ref_citation,
        JSON_EXTRACT(
              works,
              "$.articles.article.back.ref-list.ref.element-citation.person-group.name") AS authors,
        1 AS reference_order,
    FROM `multiobs.publicdb_scielo_2026_09.works_json`
    WHERE JSON_EXTRACT_ARRAY(
              works,
              "$.articles.article.back.ref-list.ref") IS NULL
);

DROP TABLE IF EXISTS `multiobs.publicdb_scielo_2026_09_rm.works_authors_references`;
CREATE TABLE `multiobs.publicdb_scielo_2026_09_rm.works_authors_references` AS (
  WITH aux AS (
    SELECT id,
      ref_citation,
      reference_order,
      JSON_EXTRACT_ARRAY(authors, "$") AS authors_a
    FROM `multiobs.publicdb_scielo_2026_09_rm.authors_references`
    WHERE JSON_EXTRACT_ARRAY(authors, "$") IS NOT NULL
  )
  SELECT id,
    ref_citation,
    reference_order,
    TRIM(JSON_EXTRACT(author,"$.surname"),'"') AS surname,
    TRIM(JSON_EXTRACT(author,"$.given-names"),'"') AS given_names,
    off+1 AS author_reference_order
  FROM aux,
  UNNEST(authors_a) author WITH OFFSET AS off
  WHERE author IS NOT NULL
    AND TRIM(JSON_EXTRACT(author,"$.surname"),'"') IS NOT NULL
  UNION ALL
  SELECT id,
    ref_citation,
    reference_order,
    TRIM(JSON_EXTRACT(authors,"$.surname"),'"') AS surname,
    TRIM(JSON_EXTRACT(authors,"$.given-names"),'"') AS given_names,
    1 AS author_reference_order
  FROM `multiobs.publicdb_scielo_2026_09_rm.authors_references`
  WHERE JSON_EXTRACT_ARRAY(authors, "$") IS NULL
    AND authors IS NOT NULL
    AND TRIM(JSON_EXTRACT(authors,"$.surname"),'"') IS NOT NULL

);

DROP TABLE IF EXISTS `multiobs.publicdb_scielo_2026_09_rm.authors_references`;

Query is running:   0%|          |

""


In [24]:
%%bigquery --project=multiobs

SELECT column_name
FROM multiobs.publicdb_scielo_2026_09_rm.INFORMATION_SCHEMA.COLUMNS
WHERE table_name LIKE 'works_authors_references'

Query is running:   0%|          |

Downloading:   0%|          |

,column_name
0,id
1,ref_citation
2,reference_order
3,surname
4,given_names
5,author_reference_order


## Keywords

In [25]:
%%bigquery --project=multiobs

DROP TABLE IF EXISTS `multiobs.publicdb_scielo_2026_09_rm.keywords`;
CREATE TABLE `multiobs.publicdb_scielo_2026_09_rm.keywords` AS (
  WITH kwd_groups AS (
    SELECT TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.unique-article-id['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id[0]['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id['#text']")
        ),'"') AS id,
      JSON_EXTRACT_ARRAY(
        works,
        "$.articles.article.front.article-meta.kwd-group") AS keywords
    FROM `multiobs.publicdb_scielo_2026_09.works_json`
    WHERE JSON_EXTRACT_ARRAY(
        works,
        "$.articles.article.front.article-meta.kwd-group") IS NOT NULL
  )
  SELECT id,
    TRIM(JSON_EXTRACT(kwds,"$['@lang_id']"),'"') AS lang_id,
    TRIM(JSON_EXTRACT(kwds,"$['@kwd-group-type']"),'"') AS type,
    JSON_EXTRACT(kwds,"$.kwd") AS keywords
  FROM kwd_groups,
  UNNEST(keywords) kwds
  WHERE JSON_EXTRACT(kwds,"$.kwd") IS NOT NULL
  UNION ALL
  SELECT TRIM(COALESCE(
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.unique-article-id['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id[0]['#text']"),
          JSON_EXTRACT(
            works,
            "$.articles.article.front.article-meta.article-id['#text']")
        ),'"') AS id,
      TRIM(JSON_EXTRACT(
        works,
        "$.articles.article.front.article-meta.kwd-group['@lang_id']"),'"') AS lang_id,
      TRIM(JSON_EXTRACT(
        works,
        "$.articles.article.front.article-meta.kwd-group['@kwd-group-type']"),'"') AS type,
      JSON_EXTRACT(
        works,
        "$.articles.article.front.article-meta.kwd-group.kwd") AS keywords,
  FROM `multiobs.publicdb_scielo_2026_09.works_json`
  WHERE JSON_EXTRACT_ARRAY(
        works,
        "$.articles.article.front.article-meta.kwd-group") IS NULL
        AND JSON_EXTRACT(
        works,
        "$.articles.article.front.article-meta.kwd-group.kwd") IS NOT NULL
);


DROP TABLE IF EXISTS `multiobs.publicdb_scielo_2026_09_rm.works_keywords`;
CREATE TABLE `multiobs.publicdb_scielo_2026_09_rm.works_keywords` AS (
  WITH aux AS (
    SELECT id,
      lang_id,
      type,
      JSON_EXTRACT_ARRAY(keywords,"$") AS keywords_a,
    FROM `multiobs.publicdb_scielo_2026_09_rm.keywords`
    WHERE JSON_EXTRACT_ARRAY(keywords,"$") IS NOT NULL
  )
  SELECT id,
    lang_id,
    type,
    TRIM(keyword,'"') AS keyword,
    off+1 AS keyword_order
  FROM aux,
  UNNEST(keywords_a) keyword WITH OFFSET as off
  WHERE keyword IS NOT NULL
  UNION ALL
  SELECT id,
    lang_id,
    type,
    TRIM(keywords,'"') AS keyword,
    1 AS keyword_order
  FROM `multiobs.publicdb_scielo_2026_09_rm.keywords`
  WHERE JSON_EXTRACT_ARRAY(keywords,"$") IS NULL
    AND TRIM(keywords,'"') IS NOT NULL
);

DROP TABLE IF EXISTS `multiobs.publicdb_scielo_2026_09_rm.keywords`;

Query is running:   0%|          |

""


In [26]:
%%bigquery --project=multiobs

SELECT column_name
FROM multiobs.publicdb_scielo_2026_09_rm.INFORMATION_SCHEMA.COLUMNS
WHERE table_name LIKE 'works_keywords'

Query is running:   0%|          |

Downloading:   0%|          |

,column_name
0,id
1,lang_id
2,type
3,keyword
4,keyword_order


## Tables

In [ ]:
%%bigquery --project=multiobs

SELECT table_name
FROM multiobs.publicdb_scielo_2026_09_rm.INFORMATION_SCHEMA.TABLES
WHERE table_name LIKE '%'

Query is running:   0%|          |

Downloading:   0%|          |

,table_name
0,works_keywords
1,works
2,works_authors
3,works_subjects
4,works_references
5,journals
6,works_authors_references
7,works_biblio
8,works_affiliations
9,works_abstracts
